训练代码（记得改路径）

In [ ]:
#coding:utf-8
import os
import sys
from ultralytics import YOLO
import torch
import warnings

warnings.filterwarnings('ignore')

# 🔥 小麦病虫害项目根目录（固定）
ROOT_DIR = r"D:\微信QQ\FireDetection"
os.chdir(ROOT_DIR)

# ===================== 修复 __file__ 报错的核心 =====================
def get_abs_path(relative_path):
    # 直接用根目录拼接，彻底解决 __file__ 报错
    abs_path = os.path.join(ROOT_DIR, relative_path)
    return os.path.normpath(abs_path)

def check_file_exists(file_path):
    if os.path.exists(file_path):
        print(f"✅ 找到文件：{file_path}")
        return True
    else:
        print(f"❌ 未找到文件：{file_path}")
        return False

def print_gpu_memory_info():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1e6
        reserved = torch.cuda.memory_reserved(0) / 1e6
        max_allocated = torch.cuda.max_memory_allocated(0) / 1e6

        print(f"\n📊 GPU内存使用情况：")
        print(f"  已分配: {allocated:.0f} MB")
        print(f"  已预留: {reserved:.0f} MB")
        print(f"  最大分配: {max_allocated:.0f} MB")
        torch.cuda.empty_cache()
    else:
        print("\n⚠️  未检测到GPU，将使用CPU训练")

def train_xiaomai_disease():
    print(f"\n📌 小麦病虫害检测训练")
    print(f"📌 项目根目录：{os.getcwd()}")

    # 数据集路径
    data_yaml_abs = get_abs_path("datasets\data.yaml")

    if not check_file_exists(data_yaml_abs):
        print("\n❌ 数据集配置文件不存在，退出训练")
        sys.exit(1)

    device = 0 if torch.cuda.is_available() else 'cpu'
    print(f"\n📌 使用设备：{device}")

    print("\n🔧 加载YOLOv8n预训练模型...")
    model = YOLO("yolov8n.pt")

    print("\n📊 初始显存使用：")
    print_gpu_memory_info()

    # 训练参数
    train_config = {
        'data': data_yaml_abs,
        'epochs': 50,#训练轮数
        'batch': 16,
        'imgsz': 640,
        'device': device,
        'augment': True,
        'degrees': 60,
        'fliplr': 0.5,
        'flipud': 0.3,
        'scale': 0.6,
        'translate': 0.15,
        'perspective': 0.001,
        'patience': 10,
        'workers': 0 if device == 'cpu' else 2,
        'save': True,
        'cache': 'ram',
        'amp': torch.cuda.is_available(),
        'verbose': True,
        'project': 'runs/detect',
        'name': 'xiaomai_model',
        'exist_ok': True,
    }

    print("\n🚀 开始训练小麦病虫害检测模型...")
    try:
        results = model.train(**train_config)
        print("\n✅ 训练完成！")

        # 导出 ONNX
        print("\n🔄 导出模型为 ONNX 格式...")
        model.export(format='onnx', device='cpu', imgsz=640, simplify=True)
        print("✅ ONNX 导出成功！")

    except Exception as e:
        print(f"\n❌ 训练出错：{str(e)[:150]}")

if __name__ == '__main__':
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(42)

    train_xiaomai_disease()
    print("\n🎉 小麦病虫害模型训练全部完成！")

对图像进行检测

In [ ]:
# 🔥 Jupyter 必须加这一行，才能显示图像
%matplotlib inline

from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

# ==================== 强制中文正常显示 ====================
plt.rcParams['font.sans-serif'] = ['SimHei']  # 黑体
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# ==================== 模型与图片路径 ====================
path = r'D:\微信QQ\FireDetection\runs\detect\runs\detect\xiaomai_model\weights\best.pt'
img_path1 = r'D:\微信QQ\FireDetection\9.png'
img_path2 = r'D:\微信QQ\FireDetection\9-2.png'

# 加载模型
model = YOLO(path, task='detect')

# ==================== 检测 9.PNG ====================
print("正在检测：图片一")
results1 = model(img_path1)
img1 = results1[0].plot()
img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)

# ==================== 检测 9-2.PNG ====================
print("正在检测：图片二")
results2 = model(img_path2)
img2 = results2[0].plot()
img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

# ==================== 强制显示两张图 ====================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

ax1.imshow(img1)
ax1.set_title("小麦病虫害检测 图片1", fontsize=14)
ax1.axis("off")

ax2.imshow(img2)
ax2.set_title("小麦病虫害检测 图片2", fontsize=14)
ax2.axis("off")

plt.tight_layout()
plt.show()  # 强制刷新显示

对摄像头进行检测

In [ ]:
# 🔥 Jupyter 必须加这一行，才能显示图像
%matplotlib inline

from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
from IPython.display import clear_output
import numpy as np

# ==================== 强制中文正常显示 ====================
plt.rcParams['font.sans-serif'] = ['SimHei']  # 黑体
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)

# ==================== 模型路径 ====================
model_path = r'D:\微信QQ\FireDetection\runs\detect\runs\detect\xiaomai_model\weights\best.pt'

# 加载模型（conf=0.4 减少误检）
model = YOLO(model_path, task='detect')

# ==================== 打开摄像头 ====================
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("摄像头打开失败")
    exit()

print("摄像头已启动 → 按 q 键停止")

# ==================== Jupyter 实时显示循环 ====================
try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("读取失败")
            break

        # YOLO 检测
        results = model(frame)
        frame_det = results[0].plot()

        # BGR → RGB（matplotlib 必须转）
        frame_rgb = cv2.cvtColor(frame_det, cv2.COLOR_BGR2RGB)

        # ==================== 强制刷新 matplotlib 显示 ====================
        clear_output(wait=True)  # 关键：清屏，实现视频流畅效果
        plt.imshow(frame_rgb)
        plt.title("小麦病虫害实时检测", fontsize=16)
        plt.axis("off")
        plt.tight_layout()
        plt.show()

        # 按 q 退出
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    # 无论如何都释放摄像头
    cap.release()
    cv2.destroyAllWindows()
    print("已停止检测")